# Likelihoods across chromosomes of different lengths

Real data cover many chromosomes, with map lengths ranging from about 0.7 M (chromosome 22) to
about 2.8 M (chromosome 1, sex-averaged). This example sets up the likelihood for two chromosomes
of different lengths. Each chromosome in each gamete is one "meiosis" in `cointf`'s sense, with its
own length $L$, and given the parameters the chromosomes are independent, so their
log-likelihoods add.

In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import chi2

from cointf import Meioses, loglik_fast, simulate_meioses

## Data

1,000 gametes, each with crossovers on chromosome 1 ($L = 2.8$ M) and chromosome 22
($L = 0.7$ M), simulated with the same $\nu = 4$ and $p = 0.05$.

In [2]:
lengths = {"chr1": 2.8, "chr22": 0.7}             # Morgans
nu_true, p_true, n = 4.0, 0.05, 1000
data = {
    "chr1": simulate_meioses(L=lengths["chr1"], nu=nu_true, p=p_true, n=n, seed=1),
    "chr22": simulate_meioses(L=lengths["chr22"], nu=nu_true, p=p_true, n=n, seed=22),
}
for chrom, d in data.items():
    counts = np.array([len(x) for x in d])
    print(f"{chrom}: L = {lengths[chrom]} M, mean {counts.mean():.2f} crossovers per meiosis, "
          f"{np.mean(counts == 0):.0%} with none")

chr1: L = 2.8 M, mean 2.85 crossovers per meiosis, 3% with none
chr22: L = 0.7 M, mean 0.69 crossovers per meiosis, 43% with none


## Two ways to set up the likelihood

**One `Meioses` per chromosome.** Each carries its own scalar length; the total is the sum.

In [3]:
per_chrom = {chrom: Meioses(d, lengths[chrom]) for chrom, d in data.items()}
ll_by_chrom = {chrom: loglik_fast(m, None, nu=nu_true, p=p_true) for chrom, m in per_chrom.items()}
for chrom, ll in ll_by_chrom.items():
    print(f"{chrom}: {ll:.3f}")
print(f"total: {sum(ll_by_chrom.values()):.3f}")

chr1: -2503.896
chr22: -634.477
total: -3138.374


**One `Meioses` for everything.** Concatenate the data and pass one length per entry. This is
convenient when parameters vary between chromosomes or individuals, because `nu` and `p` can then
be arrays aligned with the same entries.

In [4]:
all_data = data["chr1"] + data["chr22"]
chrom_of = np.repeat(["chr1", "chr22"], n)          # chromosome of each entry
L_all = np.array([lengths[c] for c in chrom_of])
mz = Meioses(all_data, L_all)
print(mz)
print(f"total: {loglik_fast(mz, None, nu=nu_true, p=p_true):.3f}")

Meioses(M=2000, crossovers=3544, max_count=8)
total: -3138.374


With `per_meiosis=True` the result has one term per chromosome per gamete. Adding the two
chromosomes of each gamete gives a per-gamete log-likelihood.

In [5]:
terms = loglik_fast(mz, None, nu=nu_true, p=p_true, per_meiosis=True)
per_gamete = terms[:n] + terms[n:]
print(f"{terms.size} terms -> {per_gamete.size} gametes; first three: {per_gamete[:3].round(3)}")

2000 terms -> 1000 gametes; first three: [-3.581 -1.903 -3.377]


## Joint fit

The pooled maximum-likelihood fit of $(\nu, p)$ uses both chromosomes, on the scale
$(\log \nu, \operatorname{logit} p)$.

In [6]:
def fit(negloglik, x0):
    res = minimize(negloglik, x0=x0, method="Nelder-Mead", options=dict(xatol=1e-6, fatol=1e-8))
    return res.x, -res.fun


shared, ll_shared = fit(lambda t: -loglik_fast(mz, None, nu=np.exp(t[0]), p=expit(t[1])),
                        [np.log(2.0), -2.0])
print(f"shared fit: nu = {np.exp(shared[0]):.2f}, p = {expit(shared[1]):.3f}, log-likelihood {ll_shared:.3f}")

shared fit: nu = 3.72, p = 0.035, log-likelihood -3137.290


## Does interference differ between chromosomes?

Give each chromosome its own $\nu$ (sharing $p$) by passing a per-entry array, and compare with the
shared fit by a likelihood-ratio test on one degree of freedom. The data were simulated with a
shared $\nu$, so the test should usually not reject.

In [7]:
is_chr1 = chrom_of == "chr1"


def negloglik_by_chrom(t):
    nu = np.where(is_chr1, np.exp(t[0]), np.exp(t[1]))
    return -loglik_fast(mz, None, nu=nu, p=expit(t[2]))


separate, ll_separate = fit(negloglik_by_chrom, [shared[0], shared[0], shared[1]])
stat = 2 * (ll_separate - ll_shared)
print(f"chromosome-specific fit: nu_chr1 = {np.exp(separate[0]):.2f}, nu_chr22 = {np.exp(separate[1]):.2f}, "
      f"p = {expit(separate[2]):.3f}")
print(f"LR = {stat:.2f}, p-value = {chi2.sf(stat, 1):.2f}")

chromosome-specific fit: nu_chr1 = 3.75, nu_chr22 = 3.60, p = 0.035
LR = 0.10, p-value = 0.75


## Notes

- **More chromosomes.** The same pattern extends to all autosomes: concatenate the data and give
  each entry its chromosome's length (sex-specific lengths if the parents' sexes are known).
- **Short chromosomes.** On a 0.7 M chromosome most meioses have zero or one crossover, which say
  little about $\nu$ beyond the count distribution. Most of the information on interference comes
  from the longer chromosomes, which have more pairs of crossovers.
- **Mixing with covariates.** The per-entry arrays combine with the regression in
  [the age-effects example](age_effects.ipynb): compute $\nu_i$ and $p_i$ from each entry's
  parent and chromosome, then make one call.